# 14 — Finalisasi Model Klasifikasi Jenis Awan

Notebook ini digunakan untuk memfinalisasi model klasifikasi jenis awan setelah
seluruh proses training, validasi, pengujian, evaluasi, analisis kesalahan, dan
prediksi citra baru selesai dilakukan.

Finalisasi model mencakup:

1. Membaca konfigurasi model dari tahap sebelumnya.
2. Memilih hasil analisis kesalahan terbaru yang lengkap.
3. Mengikuti hubungan artefak dari analisis kesalahan ke hasil evaluasi.
4. Menggunakan checkpoint yang tercatat pada hasil evaluasi.
5. Memeriksa konsistensi model, kelas, ukuran input, dan checkpoint.
6. Memeriksa status kriteria penerimaan model.
7. Menghitung hash seluruh artefak sumber.
8. Memuat checkpoint terbaik tanpa melakukan training ulang.
9. Menjalankan smoke test menggunakan tensor sintetis.
10. Membentuk checkpoint final tanpa optimizer dan scheduler.
11. Menyalin konfigurasi preprocessing ke paket model.
12. Menyusun model manifest dan model card.
13. Memverifikasi kembali model final yang telah disimpan.
14. Menyimpan laporan finalisasi secara terstruktur.

Notebook ini tidak membaca citra train, validation, atau test. Data test tidak
digunakan untuk memilih checkpoint maupun menyesuaikan hyperparameter.

## Mengimpor Library

Library yang digunakan hanya untuk membaca metadata, memeriksa checkpoint,
membangun model, menghitung hash, serta menyimpan paket model final.

Albumentations tidak perlu dimuat karena file `eval_transform.json` hanya disalin
ke paket model. Pipeline tersebut sudah divalidasi pada notebook prediksi.

In [1]:
from pathlib import Path
from datetime import datetime
from collections import OrderedDict

import hashlib
import json
import re
import shutil
import sys
import warnings
import torchvision

try:
    import pandas as pd
    import torch

    from IPython.display import display

except ImportError as exc:
    raise ImportError(
        "Dependency finalisasi model belum lengkap.\n\n"
        "Aktifkan .venv, kemudian jalankan melalui terminal VS Code:\n\n"
        r".\.venv\Scripts\python.exe -m pip install -r requirements.txt"
    ) from exc


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.precision", 6)


print(f"Python      : {sys.version.split()[0]}")
print(f"Pandas      : {pd.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
print("Mode        : Finalisasi model tanpa training")

Python      : 3.11.9
Pandas      : 3.0.5
PyTorch     : 2.11.0+cu128
CUDA        : True
Mode        : Finalisasi model tanpa training


## Menentukan Lokasi Project

Lokasi project ditentukan secara dinamis agar notebook dapat dijalankan melalui
VS Code dari folder utama project maupun dari folder `notebooks`.

Notebook tidak menggunakan path absolut Docker `/app`.

In [2]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR


DATASET_DIR = PROJECT_DIR / "dataset"
PROCESSED_DIR = DATASET_DIR / "processed"

MODELS_DIR = PROJECT_DIR / "models"
LOGS_DIR = PROJECT_DIR / "logs"

MODEL_CONFIG_PATH = (
    PROCESSED_DIR
    / "model_config.json"
)

EVAL_TRANSFORM_PATH = (
    PROCESSED_DIR
    / "eval_transform.json"
)


path_table = pd.DataFrame({
    "Nama": [
        "PROJECT_DIR",
        "PROCESSED_DIR",
        "MODELS_DIR",
        "LOGS_DIR",
        "MODEL_CONFIG_PATH",
        "EVAL_TRANSFORM_PATH",
    ],
    "Path": [
        PROJECT_DIR,
        PROCESSED_DIR,
        MODELS_DIR,
        LOGS_DIR,
        MODEL_CONFIG_PATH,
        EVAL_TRANSFORM_PATH,
    ],
})

path_table["Ada"] = (
    path_table["Path"]
    .map(Path.exists)
)

display(path_table)

,Nama,Path,Ada
0,PROJECT_DIR,D:\n8n-logsiswaparalayang\cloud-classification,True
1,PROCESSED_DIR,D:\n8n-logsiswaparalayang\cloud-classification\dataset\processed,True
2,MODELS_DIR,D:\n8n-logsiswaparalayang\cloud-classification\models,True
3,LOGS_DIR,D:\n8n-logsiswaparalayang\cloud-classification\logs,True
4,MODEL_CONFIG_PATH,D:\n8n-logsiswaparalayang\cloud-classification\dataset\processed\model_config.json,True
5,EVAL_TRANSFORM_PATH,D:\n8n-logsiswaparalayang\cloud-classification\dataset\processed\eval_transform.json,True


In [3]:
SRC_DIR = (
    PROJECT_DIR
    / "src"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )


from model_factory import (
    create_cloud_classifier,
)

In [4]:
required_directories = [
    PROCESSED_DIR,
    MODELS_DIR,
    LOGS_DIR,
]

required_files = [
    MODEL_CONFIG_PATH,
    EVAL_TRANSFORM_PATH,
]


missing_directories = [
    path
    for path in required_directories
    if not path.is_dir()
]

missing_files = [
    path
    for path in required_files
    if not path.is_file()
]


if missing_directories:
    raise FileNotFoundError(
        "Folder wajib tidak ditemukan:\n- "
        + "\n- ".join(
            str(path)
            for path in missing_directories
        )
    )


if missing_files:
    raise FileNotFoundError(
        "Artefak wajib tidak ditemukan:\n- "
        + "\n- ".join(
            str(path)
            for path in missing_files
        )
    )


print("Seluruh prasyarat dasar finalisasi tersedia.")

Seluruh prasyarat dasar finalisasi tersedia.


## Konfigurasi Finalisasi

`RELEASE_VERSION` merupakan versi paket model final.

Status paket akan ditentukan otomatis:

- `approved`: seluruh kriteria penerimaan yang ditetapkan terpenuhi.
- `research_candidate`: kriteria penerimaan belum ditetapkan.
- `blocked`: terdapat kriteria penerimaan yang tidak terpenuhi.

Model berstatus `blocked` tidak akan dikemas. Model berstatus
`research_candidate` tetap dapat dikemas untuk kebutuhan penelitian, tetapi
tidak boleh dianggap sebagai model production yang telah disetujui.

In [5]:
RELEASE_VERSION = "1.0.0"

SELECTED_ERROR_ANALYSIS_RUN = None

ALLOW_RESEARCH_CANDIDATE = True

EXPORT_TORCHSCRIPT = False

SMOKE_TEST_BATCH_SIZE = 2


if not re.fullmatch(
    r"[0-9]+\.[0-9]+\.[0-9]+",
    RELEASE_VERSION,
):
    raise ValueError(
        "RELEASE_VERSION harus menggunakan format semantic version, "
        "misalnya 1.0.0."
    )


if SMOKE_TEST_BATCH_SIZE < 1:
    raise ValueError(
        "SMOKE_TEST_BATCH_SIZE minimal bernilai 1."
    )


print(f"Release version          : {RELEASE_VERSION}")
print(f"Izinkan research candidate: {ALLOW_RESEARCH_CANDIDATE}")
print(f"Export TorchScript       : {EXPORT_TORCHSCRIPT}")

Release version          : 1.0.0
Izinkan research candidate: True
Export TorchScript       : False


In [6]:
def load_json(
    file_path: Path,
) -> dict:
    """
    Membaca file JSON dan memastikan hasilnya berupa dictionary.
    """
    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        content = json.load(file)

    if not isinstance(content, dict):
        raise TypeError(
            f"Isi {file_path.name} harus berupa dictionary."
        )

    return content


model_config = load_json(
    MODEL_CONFIG_PATH
)

eval_transform_config = load_json(
    EVAL_TRANSFORM_PATH
)


required_model_keys = {
    "model_name",
    "num_classes",
    "class_to_idx",
    "input_shape",
}


missing_model_keys = (
    required_model_keys
    - set(model_config.keys())
)


if missing_model_keys:
    raise KeyError(
        "model_config.json belum lengkap. Kunci yang hilang:\n- "
        + "\n- ".join(
            sorted(missing_model_keys)
        )
    )


MODEL_NAME = str(
    model_config["model_name"]
)

NUM_CLASSES = int(
    model_config["num_classes"]
)

CLASS_TO_IDX = {
    str(class_name): int(class_idx)
    for class_name, class_idx
    in model_config["class_to_idx"].items()
}

CLASS_TO_IDX = dict(
    sorted(
        CLASS_TO_IDX.items(),
        key=lambda item: item[1],
    )
)

IDX_TO_CLASS = {
    class_idx: class_name
    for class_name, class_idx
    in CLASS_TO_IDX.items()
}


INPUT_SHAPE = tuple(
    map(
        int,
        model_config["input_shape"],
    )
)


if len(INPUT_SHAPE) != 3:
    raise ValueError(
        "input_shape harus menggunakan format "
        "[channels, height, width]."
    )


INPUT_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH = (
    INPUT_SHAPE
)

DROPOUT_RATE = float(
    model_config.get(
        "dropout_rate",
        0.0,
    )
)

SAFE_MODEL_NAME = re.sub(
    r"[^A-Za-z0-9_.-]+",
    "_",
    MODEL_NAME,
)


expected_indices = list(
    range(NUM_CLASSES)
)

actual_indices = list(
    CLASS_TO_IDX.values()
)


if actual_indices != expected_indices:
    raise ValueError(
        "Indeks kelas tidak berurutan mulai dari 0.\n"
        f"Ditemukan: {actual_indices}"
    )


if len(CLASS_TO_IDX) != NUM_CLASSES:
    raise ValueError(
        "Jumlah kelas pada class_to_idx tidak sama dengan num_classes."
    )


model_identity_df = pd.DataFrame({
    "Parameter": [
        "Model",
        "Jumlah kelas",
        "Input channels",
        "Image height",
        "Image width",
        "Dropout rate",
    ],
    "Nilai": [
        MODEL_NAME,
        NUM_CLASSES,
        INPUT_CHANNELS,
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        DROPOUT_RATE,
    ],
})

display(model_identity_df)

,Parameter,Nilai
0,Model,resnet50
1,Jumlah kelas,7
2,Input channels,3
3,Image height,224
4,Image width,224
5,Dropout rate,0.3


In [7]:
class_mapping_df = pd.DataFrame({
    "class_idx": list(
        IDX_TO_CLASS.keys()
    ),
    "class_name": list(
        IDX_TO_CLASS.values()
    ),
})

display(class_mapping_df)

print(f"Jumlah kelas terverifikasi: {len(class_mapping_df)}")

,class_idx,class_name
0,0,1_cumulus
1,1,2_altocumulus
2,2,3_cirrus
3,3,4_clearsky
4,4,5_stratocumulus
5,5,6_cumulonimbus
6,6,7_mixed


Jumlah kelas terverifikasi: 7


## Memilih Run Analisis Kesalahan

Finalisasi dimulai dari hasil analisis kesalahan terbaru. Selanjutnya, notebook
mengikuti informasi `selected_runs.evaluation` di dalam ringkasan analisis
kesalahan.

Pendekatan ini memastikan hasil evaluasi dan analisis kesalahan berasal dari
rangkaian eksperimen yang sama.

In [8]:
def select_complete_run(
    parent_directory: Path,
    pattern: str,
    required_files: list[str],
    selected_run: str | None = None,
) -> Path:
    """
    Memilih folder run yang memiliki seluruh artefak wajib.

    Jika selected_run diisi, folder tersebut digunakan secara eksplisit.
    Jika tidak, run lengkap terbaru dipilih berdasarkan waktu modifikasi.
    """
    if selected_run is not None:
        selected_path = Path(
            selected_run
        )

        if not selected_path.is_absolute():
            selected_path = (
                parent_directory
                / selected_path
            )

        candidate_directories = [
            selected_path.resolve()
        ]

    else:
        candidate_directories = sorted(
            [
                path.resolve()
                for path
                in parent_directory.glob(pattern)
                if path.is_dir()
            ],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )


    complete_runs = []

    for run_directory in candidate_directories:
        run_complete = all(
            (
                run_directory
                / required_file
            ).is_file()
            for required_file
            in required_files
        )

        if run_complete:
            complete_runs.append(
                run_directory
            )


    if not complete_runs:
        raise FileNotFoundError(
            "Tidak ditemukan run lengkap untuk pola "
            f"'{pattern}'.\n"
            "Pastikan notebook tahap sebelumnya sudah selesai dijalankan."
        )


    return complete_runs[0]


def resolve_project_artifact(
    relative_or_absolute_path: str | Path,
) -> Path:
    """
    Mengubah path artefak menjadi path absolut dan memastikan
    artefak masih berada di dalam project.
    """
    artifact_path = Path(
        relative_or_absolute_path
    )

    if not artifact_path.is_absolute():
        artifact_path = (
            PROJECT_DIR
            / artifact_path
        )

    artifact_path = artifact_path.resolve()


    try:
        artifact_path.relative_to(
            PROJECT_DIR
        )

    except ValueError as exc:
        raise ValueError(
            "Artefak berada di luar folder project:\n"
            f"{artifact_path}"
        ) from exc


    return artifact_path

In [9]:
ERROR_ANALYSIS_REQUIRED_FILES = [
    "error_analysis_summary.json",
    "error_analysis_report.md",
    "misclassified_samples.csv",
    "per_class_error_profile.csv",
]


ERROR_ANALYSIS_RUN_DIR = select_complete_run(
    parent_directory=LOGS_DIR,
    pattern=(
        f"{SAFE_MODEL_NAME}_error_analysis_*"
    ),
    required_files=(
        ERROR_ANALYSIS_REQUIRED_FILES
    ),
    selected_run=(
        SELECTED_ERROR_ANALYSIS_RUN
    ),
)


ERROR_ANALYSIS_SUMMARY_PATH = (
    ERROR_ANALYSIS_RUN_DIR
    / "error_analysis_summary.json"
)

ERROR_ANALYSIS_REPORT_PATH = (
    ERROR_ANALYSIS_RUN_DIR
    / "error_analysis_report.md"
)


error_analysis_summary = load_json(
    ERROR_ANALYSIS_SUMMARY_PATH
)


print("Run analisis kesalahan yang dipilih:")
print(
    ERROR_ANALYSIS_RUN_DIR.relative_to(
        PROJECT_DIR
    )
)

Run analisis kesalahan yang dipilih:
logs\resnet50_error_analysis_20260826_234013_581950


In [10]:
selected_runs = error_analysis_summary.get(
    "selected_runs",
    {},
)

evaluation_run_value = selected_runs.get(
    "evaluation"
)


if not evaluation_run_value:
    raise KeyError(
        "error_analysis_summary.json tidak memiliki "
        "selected_runs.evaluation."
    )


EVALUATION_RUN_DIR = resolve_project_artifact(
    evaluation_run_value
)


EVALUATION_SUMMARY_PATH = (
    EVALUATION_RUN_DIR
    / "model_evaluation_summary.json"
)

EVALUATION_REPORT_PATH = (
    EVALUATION_RUN_DIR
    / "model_evaluation_report.md"
)

ACCEPTANCE_PATH = (
    EVALUATION_RUN_DIR
    / "acceptance_criteria.csv"
)


required_evaluation_files = [
    EVALUATION_SUMMARY_PATH,
    EVALUATION_REPORT_PATH,
    ACCEPTANCE_PATH,
]


missing_evaluation_files = [
    path
    for path in required_evaluation_files
    if not path.is_file()
]


if missing_evaluation_files:
    raise FileNotFoundError(
        "Artefak evaluasi tidak lengkap:\n- "
        + "\n- ".join(
            str(path)
            for path in missing_evaluation_files
        )
    )


evaluation_summary = load_json(
    EVALUATION_SUMMARY_PATH
)

acceptance_df = pd.read_csv(
    ACCEPTANCE_PATH
)


print("Run evaluasi yang terhubung:")
print(
    EVALUATION_RUN_DIR.relative_to(
        PROJECT_DIR
    )
)

display(acceptance_df)

Run evaluasi yang terhubung:
logs\resnet50_evaluation_20260826_233246_842963


,criterion,observed,operator,threshold,passed,status
0,Minimum test accuracy,0.763444,>=,NaN,NaN,Tidak ditetapkan
1,Minimum test macro F1,0.750912,>=,NaN,NaN,Tidak ditetapkan
2,Minimum recall kelas terendah,0.553333,>=,NaN,NaN,Tidak ditetapkan
3,Maksimum absolute macro F1 gap,0.141894,<=,NaN,NaN,Tidak ditetapkan
4,Maksimum test ECE,0.087396,<=,NaN,NaN,Tidak ditetapkan


## Menentukan Status Release

Notebook tidak menetapkan nilai ambang evaluasi baru. Penetapan ambang setelah
melihat hasil test dapat menimbulkan bias.

Status model ditentukan hanya dari kriteria penerimaan yang sudah tersimpan oleh
notebook evaluasi.

In [11]:
acceptance_evaluation = (
    evaluation_summary.get(
        "acceptance_evaluation",
        {},
    )
)

acceptance_criteria = (
    acceptance_evaluation.get(
        "criteria",
        [],
    )
)


evaluated_criteria = [
    criterion
    for criterion in acceptance_criteria
    if criterion.get("passed") is not None
]

failed_criteria = [
    criterion
    for criterion in evaluated_criteria
    if not bool(
        criterion.get("passed")
    )
]


if failed_criteria:
    RELEASE_STATUS = "blocked"

elif evaluated_criteria:
    RELEASE_STATUS = "approved"

else:
    RELEASE_STATUS = (
        "research_candidate"
    )


release_status_df = pd.DataFrame({
    "Parameter": [
        "Jumlah kriteria tersedia",
        "Jumlah kriteria dinilai",
        "Jumlah kriteria gagal",
        "Status release",
    ],
    "Nilai": [
        len(acceptance_criteria),
        len(evaluated_criteria),
        len(failed_criteria),
        RELEASE_STATUS,
    ],
})

display(release_status_df)


if RELEASE_STATUS == "blocked":
    failed_names = [
        str(
            criterion.get(
                "criterion",
                "Kriteria tanpa nama",
            )
        )
        for criterion
        in failed_criteria
    ]

    raise RuntimeError(
        "Model belum dapat difinalisasi karena terdapat "
        "kriteria penerimaan yang tidak terpenuhi:\n- "
        + "\n- ".join(failed_names)
    )


if (
    RELEASE_STATUS
    == "research_candidate"
    and not ALLOW_RESEARCH_CANDIDATE
):
    raise RuntimeError(
        "Kriteria penerimaan belum ditetapkan. "
        "Paket research candidate tidak diizinkan oleh konfigurasi."
    )


if RELEASE_STATUS == "research_candidate":
    warnings.warn(
        "Model akan dikemas sebagai research candidate karena "
        "kriteria penerimaan belum ditetapkan.",
        stacklevel=2,
    )


print(f"Status finalisasi: {RELEASE_STATUS}")

,Parameter,Nilai
0,Jumlah kriteria tersedia,5
1,Jumlah kriteria dinilai,0
2,Jumlah kriteria gagal,0
3,Status release,research_candidate


Status finalisasi: research_candidate


d:\n8n-logsiswaparalayang\cloud-classification\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: UserWarning: Model akan dikemas sebagai research candidate karena kriteria penerimaan belum ditetapkan.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
checkpoint_metadata = (
    evaluation_summary.get(
        "checkpoint",
        {},
    )
)

checkpoint_path_value = (
    checkpoint_metadata.get(
        "path"
    )
)

EXPECTED_CHECKPOINT_SHA256 = (
    checkpoint_metadata.get(
        "sha256"
    )
)


if not checkpoint_path_value:
    raise KeyError(
        "model_evaluation_summary.json tidak memiliki checkpoint.path."
    )


if not EXPECTED_CHECKPOINT_SHA256:
    raise KeyError(
        "model_evaluation_summary.json tidak memiliki checkpoint.sha256."
    )


SOURCE_CHECKPOINT_PATH = (
    resolve_project_artifact(
        checkpoint_path_value
    )
)


if not SOURCE_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        "Checkpoint sumber tidak ditemukan:\n"
        f"{SOURCE_CHECKPOINT_PATH}"
    )


print("Checkpoint sumber:")
print(
    SOURCE_CHECKPOINT_PATH.relative_to(
        PROJECT_DIR
    )
)

Checkpoint sumber:
models\resnet50_gcd_best.pth


In [13]:
def calculate_sha256(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """
    Menghitung SHA-256 file secara bertahap.
    """
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        while chunk := file.read(
            chunk_size
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


def load_checkpoint(
    checkpoint_path: Path,
    device: str | torch.device = "cpu",
) -> dict:
    """
    Memuat checkpoint dan mempertahankan kompatibilitas
    dengan beberapa versi PyTorch.
    """
    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=False,
        )

    except TypeError:
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
        )


    if not isinstance(
        checkpoint,
        dict,
    ):
        raise TypeError(
            "Checkpoint harus berupa dictionary."
        )


    return checkpoint


SOURCE_CHECKPOINT_SHA256 = (
    calculate_sha256(
        SOURCE_CHECKPOINT_PATH
    )
)


print(f"SHA-256 metadata : {EXPECTED_CHECKPOINT_SHA256}")
print(f"SHA-256 aktual   : {SOURCE_CHECKPOINT_SHA256}")


if (
    SOURCE_CHECKPOINT_SHA256
    != EXPECTED_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Hash checkpoint tidak sama dengan metadata evaluasi. "
        "Checkpoint mungkin telah berubah setelah pengujian."
    )


source_checkpoint = load_checkpoint(
    SOURCE_CHECKPOINT_PATH,
    device="cpu",
)

SHA-256 metadata : 70f2fbdd1aa783c10d935ccdc8a338f2521986156a14793d472d7a0a0e79b5c0
SHA-256 aktual   : 70f2fbdd1aa783c10d935ccdc8a338f2521986156a14793d472d7a0a0e79b5c0


In [14]:
required_checkpoint_keys = {
    "model_name",
    "model_state_dict",
    "class_to_idx",
    "input_shape",
}


missing_checkpoint_keys = (
    required_checkpoint_keys
    - set(source_checkpoint.keys())
)


if missing_checkpoint_keys:
    raise KeyError(
        "Checkpoint sumber belum lengkap. Kunci yang hilang:\n- "
        + "\n- ".join(
            sorted(missing_checkpoint_keys)
        )
    )


checkpoint_class_to_idx = {
    str(class_name): int(class_idx)
    for class_name, class_idx
    in source_checkpoint[
        "class_to_idx"
    ].items()
}

checkpoint_class_to_idx = dict(
    sorted(
        checkpoint_class_to_idx.items(),
        key=lambda item: item[1],
    )
)

checkpoint_input_shape = tuple(
    map(
        int,
        source_checkpoint[
            "input_shape"
        ],
    )
)


evaluation_model = (
    evaluation_summary.get(
        "model",
        {},
    )
)

evaluation_class_to_idx = {
    str(class_name): int(class_idx)
    for class_name, class_idx
    in evaluation_model.get(
        "class_to_idx",
        {},
    ).items()
}


error_model = (
    error_analysis_summary.get(
        "model",
        {},
    )
)

error_class_to_idx = {
    str(class_name): int(class_idx)
    for class_name, class_idx
    in error_model.get(
        "class_to_idx",
        {},
    ).items()
}


consistency_checks = {
    "Nama model checkpoint sesuai": (
        str(
            source_checkpoint[
                "model_name"
            ]
        )
        == MODEL_NAME
    ),

    "Pemetaan kelas checkpoint sesuai": (
        checkpoint_class_to_idx
        == CLASS_TO_IDX
    ),

    "Pemetaan kelas evaluasi sesuai": (
        evaluation_class_to_idx
        == CLASS_TO_IDX
    ),

    "Pemetaan kelas analisis kesalahan sesuai": (
        error_class_to_idx
        == CLASS_TO_IDX
    ),

    "Input shape checkpoint sesuai": (
        checkpoint_input_shape
        == INPUT_SHAPE
    ),

    "Jumlah kelas evaluasi sesuai": (
        int(
            evaluation_model.get(
                "num_classes",
                -1,
            )
        )
        == NUM_CLASSES
    ),

    "Jumlah kelas analisis kesalahan sesuai": (
        int(
            error_model.get(
                "num_classes",
                -1,
            )
        )
        == NUM_CLASSES
    ),

    "Checkpoint tidak dipilih dari test": (
        source_checkpoint.get(
            "test_data_used",
            False,
        )
        is False
    ),

    "Evaluation tidak memakai test untuk seleksi": (
        evaluation_summary.get(
            "evaluation_design",
            {},
        ).get(
            "test_used_for_model_selection",
            False,
        )
        is False
    ),

    "Analisis tidak memperbarui bobot": (
        error_analysis_summary.get(
            "analysis_design",
            {},
        ).get(
            "model_weights_updated",
            False,
        )
        is False
    ),
}


consistency_df = pd.DataFrame({
    "Pemeriksaan": (
        consistency_checks.keys()
    ),
    "Berhasil": (
        consistency_checks.values()
    ),
})

display(consistency_df)


failed_consistency_checks = [
    check_name
    for check_name, status
    in consistency_checks.items()
    if not status
]


if failed_consistency_checks:
    raise RuntimeError(
        "Konsistensi artefak gagal:\n- "
        + "\n- ".join(
            failed_consistency_checks
        )
    )


print("Seluruh artefak berasal dari model yang konsisten.")

,Pemeriksaan,Berhasil
0,Nama model checkpoint sesuai,True
1,Pemetaan kelas checkpoint sesuai,True
2,Pemetaan kelas evaluasi sesuai,True
3,Pemetaan kelas analisis kesalahan sesuai,True
4,Input shape checkpoint sesuai,True
5,Jumlah kelas evaluasi sesuai,True
6,Jumlah kelas analisis kesalahan sesuai,True
7,Checkpoint tidak dipilih dari test,True
8,Evaluation tidak memakai test untuk seleksi,True
9,Analisis tidak memperbarui bobot,True


Seluruh artefak berasal dari model yang konsisten.


In [15]:
source_artifact_paths = {
    "model_config": (
        MODEL_CONFIG_PATH
    ),
    "eval_transform": (
        EVAL_TRANSFORM_PATH
    ),
    "source_checkpoint": (
        SOURCE_CHECKPOINT_PATH
    ),
    "evaluation_summary": (
        EVALUATION_SUMMARY_PATH
    ),
    "evaluation_report": (
        EVALUATION_REPORT_PATH
    ),
    "acceptance_criteria": (
        ACCEPTANCE_PATH
    ),
    "error_analysis_summary": (
        ERROR_ANALYSIS_SUMMARY_PATH
    ),
    "error_analysis_report": (
        ERROR_ANALYSIS_REPORT_PATH
    ),
}


SOURCE_HASHES_BEFORE = {
    artifact_name: calculate_sha256(
        artifact_path
    )
    for artifact_name, artifact_path
    in source_artifact_paths.items()
}


source_hash_df = pd.DataFrame([
    {
        "artifact": artifact_name,
        "path": str(
            artifact_path.relative_to(
                PROJECT_DIR
            )
        ),
        "sha256": (
            SOURCE_HASHES_BEFORE[
                artifact_name
            ]
        ),
    }
    for artifact_name, artifact_path
    in source_artifact_paths.items()
])


display(source_hash_df)

,artifact,path,sha256
0,model_config,dataset\processed\model_config.json,2b907a24e7cddc0941805376a8ec64b1ff556d5e830fb5df09f6a3e11e9f6e50
1,eval_transform,dataset\processed\eval_transform.json,0f11f7bacb0cd31d02b3d4e6583ed9c5f2bc606be4b48c42b4d6e8baa3b5c5b9
2,source_checkpoint,models\resnet50_gcd_best.pth,70f2fbdd1aa783c10d935ccdc8a338f2521986156a14793d472d7a0a0e79b5c0
3,evaluation_summary,logs\resnet50_evaluation_20260826_233246_842963\model_evaluation_summary.json,7bb60f853452504cd0b12c371ce09201147ed2df08f3ea52edb054a7b4567e8c
4,evaluation_report,logs\resnet50_evaluation_20260826_233246_842963\model_evaluation_report.md,231362b73a7408e940b87564adcdde29173a26c6f8c1650267422d65bf3f2809
5,acceptance_criteria,logs\resnet50_evaluation_20260826_233246_842963\acceptance_criteria.csv,b661bb2631024f6318a3ff13d227e98e469d0b881b5ed91cf3a6c575c16ed0be
6,error_analysis_summary,logs\resnet50_error_analysis_20260826_234013_581950\error_analysis_summary.json,1d39779ac513b7b8acfd4a5bb014dffaea0f6d8e1e79c0f572d26c229fa68155
7,error_analysis_report,logs\resnet50_error_analysis_20260826_234013_581950\error_analysis_report.md,136ee420d4dc043c612aa74469c11a9e45b84815c1836fb5c17df60a5427dc7b


## Membangun Model dan Memuat Bobot

Model dibangun dengan `pretrained=False`. Bobot ImageNet tidak diunduh ulang
karena seluruh parameter akan dimuat dari checkpoint hasil training.

Fungsi pembentukan model ditulis satu kali dan digunakan kembali saat memeriksa
checkpoint sumber maupun checkpoint final.

In [16]:
def build_model() -> torch.nn.Module:
    """
    Membentuk arsitektur sesuai model_config.json.
    """
    model = create_cloud_classifier(
        num_classes=NUM_CLASSES,
        pretrained=False,
        dropout_rate=DROPOUT_RATE,
    )


    return model


def validate_model_output(
    model: torch.nn.Module,
    batch_size: int,
) -> dict:
    """
    Menjalankan smoke test menggunakan tensor sintetis.
    Tidak menggunakan citra train, validation, maupun test.
    """
    model = model.to("cpu")
    model.requires_grad_(False)
    model.eval()


    dummy_input = torch.zeros(
        (
            batch_size,
            INPUT_CHANNELS,
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
        dtype=torch.float32,
    )


    with torch.inference_mode():
        logits = model(
            dummy_input
        )

        probabilities = torch.softmax(
            logits,
            dim=1,
        )


    expected_output_shape = (
        batch_size,
        NUM_CLASSES,
    )


    output_checks = {
        "Output shape sesuai": (
            tuple(logits.shape)
            == expected_output_shape
        ),

        "Logits seluruhnya finite": (
            torch.isfinite(
                logits
            ).all().item()
        ),

        "Probabilitas seluruhnya finite": (
            torch.isfinite(
                probabilities
            ).all().item()
        ),

        "Jumlah probabilitas sama dengan satu": (
            torch.allclose(
                probabilities.sum(dim=1),
                torch.ones(batch_size),
                atol=1e-6,
                rtol=0.0,
            )
        ),
    }


    return {
        "checks": output_checks,
        "output_shape": list(
            logits.shape
        ),
        "logits_min": float(
            logits.min().item()
        ),
        "logits_max": float(
            logits.max().item()
        ),
        "probability_sum_min": float(
            probabilities
            .sum(dim=1)
            .min()
            .item()
        ),
        "probability_sum_max": float(
            probabilities
            .sum(dim=1)
            .max()
            .item()
        ),
        "dummy_input": dummy_input,
        "reference_logits": logits.detach().clone(),
    }

In [17]:
source_model = build_model()


load_result = source_model.load_state_dict(
    source_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)


if load_result.missing_keys:
    raise RuntimeError(
        "Parameter checkpoint yang hilang:\n"
        f"{load_result.missing_keys}"
    )


if load_result.unexpected_keys:
    raise RuntimeError(
        "Parameter checkpoint yang tidak dikenali:\n"
        f"{load_result.unexpected_keys}"
    )


source_smoke_test = validate_model_output(
    model=source_model,
    batch_size=SMOKE_TEST_BATCH_SIZE,
)


source_smoke_test_df = pd.DataFrame({
    "Pemeriksaan": (
        source_smoke_test[
            "checks"
        ].keys()
    ),
    "Berhasil": (
        source_smoke_test[
            "checks"
        ].values()
    ),
})

display(source_smoke_test_df)


if not all(
    source_smoke_test[
        "checks"
    ].values()
):
    raise RuntimeError(
        "Smoke test checkpoint sumber gagal."
    )


print(
    "Checkpoint sumber berhasil dimuat "
    "menggunakan strict=True."
)

print(
    "Output shape:",
    source_smoke_test[
        "output_shape"
    ],
)

,Pemeriksaan,Berhasil
0,Output shape sesuai,True
1,Logits seluruhnya finite,True
2,Probabilitas seluruhnya finite,True
3,Jumlah probabilitas sama dengan satu,True


Checkpoint sumber berhasil dimuat menggunakan strict=True.
Output shape: [2, 7]


## Menyiapkan Paket Model Final

Paket model disimpan di dalam folder `models/releases/`, sehingga tidak
mengubah fungsi folder utama project.

Setiap versi release menggunakan folder berbeda. Notebook tidak menimpa release
yang sudah ada. Jika versi telah digunakan, ubah nilai `RELEASE_VERSION`.

In [18]:
FINALIZATION_RUN_ID = (
    datetime.now()
    .astimezone()
    .strftime("%Y%m%d_%H%M%S_%f")
)


RELEASE_NAME = (
    f"{SAFE_MODEL_NAME}_gcd_"
    f"v{RELEASE_VERSION}"
)


RELEASES_DIR = (
    MODELS_DIR
    / "releases"
)

RELEASE_DIR = (
    RELEASES_DIR
    / RELEASE_NAME
)

FINALIZATION_OUTPUT_DIR = (
    LOGS_DIR
    / (
        f"{SAFE_MODEL_NAME}_finalization_"
        f"{FINALIZATION_RUN_ID}"
    )
)


FINAL_MODEL_PATH = (
    RELEASE_DIR
    / f"{SAFE_MODEL_NAME}_gcd_final.pth"
)

FINAL_MODEL_CONFIG_PATH = (
    RELEASE_DIR
    / "model_config.json"
)

FINAL_EVAL_TRANSFORM_PATH = (
    RELEASE_DIR
    / "eval_transform.json"
)

FINAL_CLASS_MAPPING_PATH = (
    RELEASE_DIR
    / "class_mapping.json"
)

MODEL_CARD_PATH = (
    RELEASE_DIR
    / "MODEL_CARD.md"
)

MODEL_MANIFEST_PATH = (
    RELEASE_DIR
    / "model_manifest.json"
)

TORCHSCRIPT_PATH = (
    RELEASE_DIR
    / f"{SAFE_MODEL_NAME}_gcd_torchscript.pt"
)


if RELEASE_DIR.exists():
    raise FileExistsError(
        "Folder release sudah tersedia:\n"
        f"{RELEASE_DIR}\n\n"
        "Gunakan RELEASE_VERSION yang berbeda agar release lama "
        "tidak tertimpa."
    )


RELEASE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

FINALIZATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


print("Folder release:")
print(
    RELEASE_DIR.relative_to(
        PROJECT_DIR
    )
)

print("\nFolder laporan finalisasi:")
print(
    FINALIZATION_OUTPUT_DIR.relative_to(
        PROJECT_DIR
    )
)

Folder release:
models\releases\resnet50_gcd_v1.0.0

Folder laporan finalisasi:
logs\resnet50_finalization_20260827_001248_921070


## Membentuk Checkpoint Final

Checkpoint training berisi state optimizer dan scheduler yang diperlukan untuk
melanjutkan training. Artefak tersebut tidak diperlukan untuk inferensi.

Checkpoint final hanya menyimpan bobot model dan metadata yang diperlukan untuk
rekonstruksi model. Bobot dipindahkan ke CPU agar checkpoint bersifat portabel.

In [19]:
def move_state_dict_to_cpu(
    state_dict: dict,
) -> OrderedDict:
    """
    Memindahkan seluruh tensor state_dict ke CPU.
    """
    cpu_state_dict = OrderedDict()

    for parameter_name, parameter_value in state_dict.items():
        if torch.is_tensor(
            parameter_value
        ):
            cpu_state_dict[
                parameter_name
            ] = (
                parameter_value
                .detach()
                .cpu()
                .contiguous()
            )

        else:
            cpu_state_dict[
                parameter_name
            ] = parameter_value

    return cpu_state_dict


def atomic_torch_save(
    payload: dict,
    destination: Path,
) -> None:
    """
    Menyimpan checkpoint melalui file sementara.
    """
    temporary_path = (
        destination.parent
        / f"{destination.name}.tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    temporary_path.replace(
        destination
    )


FINALIZED_AT = (
    datetime.now()
    .astimezone()
    .isoformat(timespec="seconds")
)


final_checkpoint_payload = {
    "format_version": "1.0",

    "release_version": (
        RELEASE_VERSION
    ),

    "release_status": (
        RELEASE_STATUS
    ),

    "finalized_at": (
        FINALIZED_AT
    ),

    "model_name": (
        MODEL_NAME
    ),

    "model_state_dict": (
        move_state_dict_to_cpu(
            source_checkpoint[
                "model_state_dict"
            ]
        )
    ),

    "num_classes": (
        NUM_CLASSES
    ),

    "class_to_idx": (
        CLASS_TO_IDX
    ),

    "idx_to_class": {
        str(class_idx): class_name
        for class_idx, class_name
        in IDX_TO_CLASS.items()
    },

    "input_shape": list(
        INPUT_SHAPE
    ),

    "dropout_rate": (
        DROPOUT_RATE
    ),

    "normalization": (
        model_config.get(
            "normalization"
        )
    ),

    "source_checkpoint": {
        "path": str(
            SOURCE_CHECKPOINT_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "sha256": (
            SOURCE_CHECKPOINT_SHA256
        ),
        "training_run_id": (
            source_checkpoint.get(
                "run_id"
            )
        ),
        "phase": (
            source_checkpoint.get(
                "phase"
            )
        ),
        "global_epoch": (
            source_checkpoint.get(
                "global_epoch"
            )
        ),
    },

    "evaluation": {
        "run": str(
            EVALUATION_RUN_DIR.relative_to(
                PROJECT_DIR
            )
        ),
        "test_metrics": (
            evaluation_summary.get(
                "test_metrics",
                {},
            )
        ),
        "acceptance_status": (
            acceptance_evaluation.get(
                "status"
            )
        ),
    },

    "error_analysis": {
        "run": str(
            ERROR_ANALYSIS_RUN_DIR.relative_to(
                PROJECT_DIR
            )
        ),
        "number_of_errors": (
            error_analysis_summary.get(
                "overall",
                {},
            ).get(
                "number_of_errors"
            )
        ),
        "dominant_error_pair": (
            error_analysis_summary.get(
                "dominant_error_pair"
            )
        ),
    },

    "training_performed_during_finalization": False,
    "model_weights_updated_during_finalization": False,
    "test_used_for_model_selection": False,
}


atomic_torch_save(
    payload=final_checkpoint_payload,
    destination=FINAL_MODEL_PATH,
)


print("Checkpoint final berhasil disimpan:")
print(
    FINAL_MODEL_PATH.relative_to(
        PROJECT_DIR
    )
)

Checkpoint final berhasil disimpan:
models\releases\resnet50_gcd_v1.0.0\resnet50_gcd_final.pth


In [20]:
def atomic_copy(
    source: Path,
    destination: Path,
) -> None:
    """
    Menyalin file melalui file sementara.
    """
    temporary_path = (
        destination.parent
        / f"{destination.name}.tmp"
    )

    shutil.copy2(
        source,
        temporary_path,
    )

    temporary_path.replace(
        destination
    )


def write_json(
    payload: dict,
    destination: Path,
) -> None:
    """
    Menyimpan dictionary sebagai JSON secara atomik.
    """
    temporary_path = (
        destination.parent
        / f"{destination.name}.tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )

    temporary_path.replace(
        destination
    )


atomic_copy(
    MODEL_CONFIG_PATH,
    FINAL_MODEL_CONFIG_PATH,
)

atomic_copy(
    EVAL_TRANSFORM_PATH,
    FINAL_EVAL_TRANSFORM_PATH,
)


final_class_mapping = {
    "num_classes": NUM_CLASSES,
    "class_to_idx": CLASS_TO_IDX,
    "idx_to_class": {
        str(class_idx): class_name
        for class_idx, class_name
        in IDX_TO_CLASS.items()
    },
}


write_json(
    final_class_mapping,
    FINAL_CLASS_MAPPING_PATH,
)


print("Konfigurasi paket model berhasil disimpan:")
print(f"- {FINAL_MODEL_CONFIG_PATH.name}")
print(f"- {FINAL_EVAL_TRANSFORM_PATH.name}")
print(f"- {FINAL_CLASS_MAPPING_PATH.name}")

Konfigurasi paket model berhasil disimpan:
- model_config.json
- eval_transform.json
- class_mapping.json


In [21]:
saved_final_checkpoint = load_checkpoint(
    FINAL_MODEL_PATH,
    device="cpu",
)


required_final_keys = {
    "format_version",
    "release_version",
    "release_status",
    "model_name",
    "model_state_dict",
    "num_classes",
    "class_to_idx",
    "input_shape",
}


missing_final_keys = (
    required_final_keys
    - set(saved_final_checkpoint.keys())
)


if missing_final_keys:
    raise KeyError(
        "Checkpoint final belum lengkap. Kunci yang hilang:\n- "
        + "\n- ".join(
            sorted(missing_final_keys)
        )
    )


final_model = build_model()


final_load_result = (
    final_model.load_state_dict(
        saved_final_checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )
)


if final_load_result.missing_keys:
    raise RuntimeError(
        "Parameter model final yang hilang:\n"
        f"{final_load_result.missing_keys}"
    )


if final_load_result.unexpected_keys:
    raise RuntimeError(
        "Parameter model final yang tidak dikenali:\n"
        f"{final_load_result.unexpected_keys}"
    )


final_smoke_test = validate_model_output(
    model=final_model,
    batch_size=SMOKE_TEST_BATCH_SIZE,
)


with torch.inference_mode():
    final_reference_logits = final_model(
        source_smoke_test[
            "dummy_input"
        ]
    )


SOURCE_AND_FINAL_OUTPUT_EQUAL = (
    torch.allclose(
        source_smoke_test[
            "reference_logits"
        ],
        final_reference_logits,
        atol=0.0,
        rtol=0.0,
    )
)


if not all(
    final_smoke_test[
        "checks"
    ].values()
):
    raise RuntimeError(
        "Smoke test model final gagal."
    )


if not SOURCE_AND_FINAL_OUTPUT_EQUAL:
    raise RuntimeError(
        "Output checkpoint final berbeda dari checkpoint sumber."
    )


print("Checkpoint final berhasil dimuat dengan strict=True.")
print("Output checkpoint sumber dan final identik.")

Checkpoint final berhasil dimuat dengan strict=True.
Output checkpoint sumber dan final identik.


## Export TorchScript Opsional

Checkpoint `.pth` tetap menjadi artefak utama. Export TorchScript hanya
dijalankan jika `EXPORT_TORCHSCRIPT = True`.

TorchScript dapat digunakan pada lingkungan yang tidak perlu membentuk ulang
arsitektur menggunakan `timm`.

In [22]:
TORCHSCRIPT_EXPORTED = False
TORCHSCRIPT_VERIFIED = False
TORCHSCRIPT_ERROR = None


if EXPORT_TORCHSCRIPT:
    try:
        final_model = final_model.to("cpu")
        final_model.eval()

        example_input = torch.zeros(
            (
                1,
                INPUT_CHANNELS,
                IMAGE_HEIGHT,
                IMAGE_WIDTH,
            ),
            dtype=torch.float32,
        )


        with torch.inference_mode():
            reference_output = (
                final_model(
                    example_input
                )
            )


        traced_model = torch.jit.trace(
            final_model,
            example_input,
            strict=True,
        )

        traced_model = torch.jit.freeze(
            traced_model.eval()
        )

        torch.jit.save(
            traced_model,
            str(TORCHSCRIPT_PATH),
        )

        TORCHSCRIPT_EXPORTED = (
            TORCHSCRIPT_PATH.is_file()
        )


        loaded_traced_model = (
            torch.jit.load(
                str(TORCHSCRIPT_PATH),
                map_location="cpu",
            )
        )

        loaded_traced_model.eval()


        with torch.inference_mode():
            traced_output = (
                loaded_traced_model(
                    example_input
                )
            )


        TORCHSCRIPT_VERIFIED = (
            torch.allclose(
                reference_output,
                traced_output,
                atol=1e-5,
                rtol=1e-5,
            )
        )


        if not TORCHSCRIPT_VERIFIED:
            raise RuntimeError(
                "Output TorchScript berbeda dari output model PyTorch."
            )


    except Exception as exc:
        TORCHSCRIPT_ERROR = (
            f"{type(exc).__name__}: {exc}"
        )

        raise RuntimeError(
            "Export TorchScript gagal. "
            f"Detail: {TORCHSCRIPT_ERROR}"
        ) from exc


else:
    print(
        "Export TorchScript dilewati karena "
        "EXPORT_TORCHSCRIPT=False."
    )


print(f"TorchScript exported : {TORCHSCRIPT_EXPORTED}")
print(f"TorchScript verified : {TORCHSCRIPT_VERIFIED}")

Export TorchScript dilewati karena EXPORT_TORCHSCRIPT=False.
TorchScript exported : False
TorchScript verified : False


## Menyusun Model Card

Model card memberikan informasi ringkas mengenai identitas model, sumber data,
metrik, batasan penggunaan, serta status release.

Nilai metrik dibaca dari hasil evaluasi yang sudah tersimpan. Notebook tidak
menghitung ulang metrik test.

In [23]:
test_metrics = (
    evaluation_summary.get(
        "test_metrics",
        {},
    )
)

class_findings = (
    evaluation_summary.get(
        "class_findings",
        {},
    )
)

lowest_test_f1 = (
    class_findings.get(
        "lowest_test_f1",
        {},
    )
)

error_overall = (
    error_analysis_summary.get(
        "overall",
        {},
    )
)

dominant_error_pair = (
    error_analysis_summary.get(
        "dominant_error_pair"
    )
)


def format_percentage(
    value,
) -> str:
    if value is None:
        return "Tidak tersedia"

    return f"{float(value):.4%}"


def format_decimal(
    value,
    digits: int = 6,
) -> str:
    if value is None:
        return "Tidak tersedia"

    return f"{float(value):.{digits}f}"


if dominant_error_pair is None:
    dominant_error_description = (
        "Tidak terdapat kesalahan prediksi."
    )

else:
    dominant_error_description = (
        f"{dominant_error_pair.get('true_class')} → "
        f"{dominant_error_pair.get('predicted_class')} "
        f"sebanyak "
        f"{int(dominant_error_pair.get('error_count', 0))} kesalahan"
    )


model_card = f"""# Model Card — {MODEL_NAME}

## Identitas Model

- Release: `{RELEASE_VERSION}`
- Status: `{RELEASE_STATUS}`
- Arsitektur: `{MODEL_NAME}`
- Framework: PyTorch
- Jumlah kelas: {NUM_CLASSES}
- Input: `{INPUT_CHANNELS} × {IMAGE_HEIGHT} × {IMAGE_WIDTH}`
- Checkpoint final: `{FINAL_MODEL_PATH.name}`
- Tanggal finalisasi: `{FINALIZED_AT}`

## Tujuan Model

Model digunakan untuk mengklasifikasikan citra awan berbasis darat ke dalam
kelas yang tercantum pada `class_mapping.json`.

## Pemetaan Kelas

{class_mapping_df.to_markdown(index=False)}

## Hasil Evaluasi Test

- Accuracy: {format_percentage(test_metrics.get("accuracy"))}
- Balanced accuracy: {format_percentage(test_metrics.get("balanced_accuracy"))}
- Precision macro: {format_percentage(test_metrics.get("precision_macro"))}
- Recall macro: {format_percentage(test_metrics.get("recall_macro"))}
- Macro F1-score: {format_percentage(test_metrics.get("f1_macro"))}
- Weighted F1-score: {format_percentage(test_metrics.get("f1_weighted"))}
- Log loss: {format_decimal(test_metrics.get("log_loss"))}

## Temuan Per Kelas

- Kelas dengan F1 test terendah: `{lowest_test_f1.get("class_name", "Tidak tersedia")}`
- F1-score kelas tersebut: {format_percentage(lowest_test_f1.get("f1_score"))}
- Recall kelas tersebut: {format_percentage(lowest_test_f1.get("recall"))}

## Analisis Kesalahan

- Jumlah sampel test: {int(error_overall.get("number_of_samples", 0)):,}
- Jumlah prediksi salah: {int(error_overall.get("number_of_errors", 0)):,}
- Error rate: {format_percentage(error_overall.get("error_rate"))}
- Pasangan kesalahan dominan: {dominant_error_description}

## Status Penerimaan

{acceptance_evaluation.get("status", "Status tidak tersedia")}

Status release model: `{RELEASE_STATUS}`.

## Preprocessing

Gunakan pipeline evaluation yang tersimpan pada `eval_transform.json`.
Transform training atau augmentasi acak tidak boleh digunakan saat inferensi.

## Batasan

1. Model hanya divalidasi pada kelas dan karakteristik data yang digunakan dalam penelitian.
2. Confidence tinggi tidak selalu berarti prediksi benar.
3. Citra dengan awan campuran dapat menghasilkan prediksi ambigu.
4. Model belum menggantikan observasi meteorologis profesional.
5. Model harus dievaluasi kembali jika digunakan pada kamera, lokasi, musim, atau kondisi pencahayaan yang berbeda.
6. Test split tidak boleh digunakan untuk memilih ulang checkpoint atau menyesuaikan hyperparameter.

## Provenance

- Checkpoint sumber: `{SOURCE_CHECKPOINT_PATH.relative_to(PROJECT_DIR)}`
- SHA-256 checkpoint sumber: `{SOURCE_CHECKPOINT_SHA256}`
- Run evaluasi: `{EVALUATION_RUN_DIR.relative_to(PROJECT_DIR)}`
- Run analisis kesalahan: `{ERROR_ANALYSIS_RUN_DIR.relative_to(PROJECT_DIR)}`

## Catatan Metodologis

Finalisasi tidak menjalankan training, tidak memperbarui bobot, dan tidak
menggunakan data test untuk pemilihan model. Smoke test hanya menggunakan tensor
sintetis bernilai nol.
"""


MODEL_CARD_PATH.write_text(
    model_card,
    encoding="utf-8",
)


print(model_card)

# Model Card — resnet50

## Identitas Model

- Release: `1.0.0`
- Status: `research_candidate`
- Arsitektur: `resnet50`
- Framework: PyTorch
- Jumlah kelas: 7
- Input: `3 × 224 × 224`
- Checkpoint final: `resnet50_gcd_final.pth`
- Tanggal finalisasi: `2026-08-27T00:12:48+07:00`

## Tujuan Model

Model digunakan untuk mengklasifikasikan citra awan berbasis darat ke dalam
kelas yang tercantum pada `class_mapping.json`.

## Pemetaan Kelas

|   class_idx | class_name      |
|------------:|:----------------|
|           0 | 1_cumulus       |
|           1 | 2_altocumulus   |
|           2 | 3_cirrus        |
|           3 | 4_clearsky      |
|           4 | 5_stratocumulus |
|           5 | 6_cumulonimbus  |
|           6 | 7_mixed         |

## Hasil Evaluasi Test

- Accuracy: 76.3444%
- Balanced accuracy: 73.9750%
- Precision macro: 78.1900%
- Recall macro: 73.9750%
- Macro F1-score: 75.0912%
- Weighted F1-score: 76.1537%
- Log loss: 0.736055

## Temuan Per Kelas

- Kelas dengan F1 test t

In [24]:
release_artifact_paths = {
    "final_checkpoint": (
        FINAL_MODEL_PATH
    ),
    "model_config": (
        FINAL_MODEL_CONFIG_PATH
    ),
    "eval_transform": (
        FINAL_EVAL_TRANSFORM_PATH
    ),
    "class_mapping": (
        FINAL_CLASS_MAPPING_PATH
    ),
    "model_card": (
        MODEL_CARD_PATH
    ),
}


if TORCHSCRIPT_EXPORTED:
    release_artifact_paths[
        "torchscript"
    ] = TORCHSCRIPT_PATH


release_artifact_hashes = {
    artifact_name: {
        "path": str(
            artifact_path.relative_to(
                PROJECT_DIR
            )
        ),
        "size_bytes": int(
            artifact_path.stat().st_size
        ),
        "sha256": calculate_sha256(
            artifact_path
        ),
    }
    for artifact_name, artifact_path
    in release_artifact_paths.items()
}


release_hash_df = pd.DataFrame([
    {
        "artifact": artifact_name,
        **artifact_metadata,
    }
    for artifact_name, artifact_metadata
    in release_artifact_hashes.items()
])


display(release_hash_df)

,artifact,path,size_bytes,sha256
0,final_checkpoint,models\releases\resnet50_gcd_v1.0.0\resnet50_gcd_final.pth,94404697,d319252c0237478770760ed00b89f89de3196133db1270c030070c6d00234d54
1,model_config,models\releases\resnet50_gcd_v1.0.0\model_config.json,2201,2b907a24e7cddc0941805376a8ec64b1ff556d5e830fb5df09f6a3e11e9f6e50
2,eval_transform,models\releases\resnet50_gcd_v1.0.0\eval_transform.json,771,0f11f7bacb0cd31d02b3d4e6583ed9c5f2bc606be4b48c42b4d6e8baa3b5c5b9
3,class_mapping,models\releases\resnet50_gcd_v1.0.0\class_mapping.json,410,2a788b86383fad595f9838e6d7ed0df40d12431ede65a9e0002f23358dd84d9d
4,model_card,models\releases\resnet50_gcd_v1.0.0\MODEL_CARD.md,2699,fcbb48f793629354d6388cfc54e604377777999d5f460977ca5b3447fbae8918


In [25]:
model_manifest = {
    "manifest_version": "1.0",

    "created_at": (
        FINALIZED_AT
    ),

    "release": {
        "name": RELEASE_NAME,
        "version": RELEASE_VERSION,
        "status": RELEASE_STATUS,
    },

    "model": {
        "model_name": MODEL_NAME,
        "framework": "PyTorch",
        "torch_version": torch.__version__,
       "model_library": (
            model_config["model_library"]
        ),

        "weights_enum": (
            model_config["weights_enum"]
        ),

        "torchvision_version": (
            torchvision.__version__
        ),
        "num_classes": NUM_CLASSES,
        "class_to_idx": CLASS_TO_IDX,
        "input_shape": list(
            INPUT_SHAPE
        ),
        "dropout_rate": DROPOUT_RATE,
    },

    "source_checkpoint": {
        "path": str(
            SOURCE_CHECKPOINT_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "sha256": (
            SOURCE_CHECKPOINT_SHA256
        ),
    },

    "source_runs": {
        "evaluation": str(
            EVALUATION_RUN_DIR.relative_to(
                PROJECT_DIR
            )
        ),
        "error_analysis": str(
            ERROR_ANALYSIS_RUN_DIR.relative_to(
                PROJECT_DIR
            )
        ),
    },

    "evaluation": {
        "test_metrics": test_metrics,
        "acceptance": (
            acceptance_evaluation
        ),
    },

    "smoke_test": {
        "batch_size": (
            SMOKE_TEST_BATCH_SIZE
        ),
        "output_shape": (
            final_smoke_test[
                "output_shape"
            ]
        ),
        "source_and_final_output_equal": (
            SOURCE_AND_FINAL_OUTPUT_EQUAL
        ),
        "checks": (
            final_smoke_test[
                "checks"
            ]
        ),
    },

    "torchscript": {
        "requested": (
            EXPORT_TORCHSCRIPT
        ),
        "exported": (
            TORCHSCRIPT_EXPORTED
        ),
        "verified": (
            TORCHSCRIPT_VERIFIED
        ),
        "error": (
            TORCHSCRIPT_ERROR
        ),
    },

    "release_artifacts": (
        release_artifact_hashes
    ),

    "methodological_controls": {
        "training_performed": False,
        "weights_updated": False,
        "validation_inference_performed": False,
        "test_inference_performed": False,
        "test_used_for_model_selection": False,
        "test_used_for_hyperparameter_tuning": False,
        "synthetic_smoke_test_only": True,
    },
}


write_json(
    model_manifest,
    MODEL_MANIFEST_PATH,
)


print("Model manifest berhasil disimpan:")
print(
    MODEL_MANIFEST_PATH.relative_to(
        PROJECT_DIR
    )
)

Model manifest berhasil disimpan:
models\releases\resnet50_gcd_v1.0.0\model_manifest.json


In [26]:
SOURCE_HASHES_AFTER = {
    artifact_name: calculate_sha256(
        artifact_path
    )
    for artifact_name, artifact_path
    in source_artifact_paths.items()
}


source_integrity_df = pd.DataFrame([
    {
        "artifact": artifact_name,
        "sha256_before": (
            SOURCE_HASHES_BEFORE[
                artifact_name
            ]
        ),
        "sha256_after": (
            SOURCE_HASHES_AFTER[
                artifact_name
            ]
        ),
        "tidak_berubah": (
            SOURCE_HASHES_BEFORE[
                artifact_name
            ]
            == SOURCE_HASHES_AFTER[
                artifact_name
            ]
        ),
    }
    for artifact_name
    in source_artifact_paths
])


display(source_integrity_df)


required_release_files = [
    FINAL_MODEL_PATH,
    FINAL_MODEL_CONFIG_PATH,
    FINAL_EVAL_TRANSFORM_PATH,
    FINAL_CLASS_MAPPING_PATH,
    MODEL_CARD_PATH,
    MODEL_MANIFEST_PATH,
]


if EXPORT_TORCHSCRIPT:
    required_release_files.append(
        TORCHSCRIPT_PATH
    )


final_checks = {
    "Konsistensi artefak sumber": (
        all(
            consistency_checks.values()
        )
    ),

    "Hash checkpoint sesuai evaluasi": (
        SOURCE_CHECKPOINT_SHA256
        == EXPECTED_CHECKPOINT_SHA256
    ),

    "Smoke test checkpoint sumber": (
        all(
            source_smoke_test[
                "checks"
            ].values()
        )
    ),

    "Smoke test checkpoint final": (
        all(
            final_smoke_test[
                "checks"
            ].values()
        )
    ),

    "Output sumber dan final identik": (
        SOURCE_AND_FINAL_OUTPUT_EQUAL
    ),

    "Artefak sumber tidak berubah": (
        source_integrity_df[
            "tidak_berubah"
        ].all()
    ),

    "Seluruh file release tersedia": all(
        path.is_file()
        for path in required_release_files
    ),

    "Status model tidak blocked": (
        RELEASE_STATUS
        != "blocked"
    ),

    "Training dijalankan": False,

    "Bobot diperbarui": False,

    "Data test dibaca ulang": False,
}


final_check_df = pd.DataFrame({
    "Pemeriksaan": (
        final_checks.keys()
    ),
    "Nilai": (
        final_checks.values()
    ),
})


negative_expected_checks = {
    "Training dijalankan",
    "Bobot diperbarui",
    "Data test dibaca ulang",
}


final_check_df["Status"] = [
    (
        "Sesuai"
        if (
            check_name
            in negative_expected_checks
            and value is False
        )
        else (
            "Berhasil"
            if value
            else "Gagal"
        )
    )
    for check_name, value
    in final_checks.items()
]


display(final_check_df)


failed_final_checks = [
    check_name
    for check_name, status
    in final_checks.items()
    if (
        check_name
        not in negative_expected_checks
        and not status
    )
]


if failed_final_checks:
    raise RuntimeError(
        "Finalisasi model gagal pada pemeriksaan berikut:\n- "
        + "\n- ".join(
            failed_final_checks
        )
    )


print("Seluruh pemeriksaan finalisasi berhasil.")

,artifact,sha256_before,sha256_after,tidak_berubah
0,model_config,2b907a24e7cddc0941805376a8ec64b1ff556d5e830fb5df09f6a3e11e9f6e50,2b907a24e7cddc0941805376a8ec64b1ff556d5e830fb5df09f6a3e11e9f6e50,True
1,eval_transform,0f11f7bacb0cd31d02b3d4e6583ed9c5f2bc606be4b48c42b4d6e8baa3b5c5b9,0f11f7bacb0cd31d02b3d4e6583ed9c5f2bc606be4b48c42b4d6e8baa3b5c5b9,True
2,source_checkpoint,70f2fbdd1aa783c10d935ccdc8a338f2521986156a14793d472d7a0a0e79b5c0,70f2fbdd1aa783c10d935ccdc8a338f2521986156a14793d472d7a0a0e79b5c0,True
3,evaluation_summary,7bb60f853452504cd0b12c371ce09201147ed2df08f3ea52edb054a7b4567e8c,7bb60f853452504cd0b12c371ce09201147ed2df08f3ea52edb054a7b4567e8c,True
4,evaluation_report,231362b73a7408e940b87564adcdde29173a26c6f8c1650267422d65bf3f2809,231362b73a7408e940b87564adcdde29173a26c6f8c1650267422d65bf3f2809,True
5,acceptance_criteria,b661bb2631024f6318a3ff13d227e98e469d0b881b5ed91cf3a6c575c16ed0be,b661bb2631024f6318a3ff13d227e98e469d0b881b5ed91cf3a6c575c16ed0be,True
6,error_analysis_summary,1d39779ac513b7b8acfd4a5bb014dffaea0f6d8e1e79c0f572d26c229fa68155,1d39779ac513b7b8acfd4a5bb014dffaea0f6d8e1e79c0f572d26c229fa68155,True
7,error_analysis_report,136ee420d4dc043c612aa74469c11a9e45b84815c1836fb5c17df60a5427dc7b,136ee420d4dc043c612aa74469c11a9e45b84815c1836fb5c17df60a5427dc7b,True


,Pemeriksaan,Nilai,Status
0,Konsistensi artefak sumber,True,Berhasil
1,Hash checkpoint sesuai evaluasi,True,Berhasil
2,Smoke test checkpoint sumber,True,Berhasil
3,Smoke test checkpoint final,True,Berhasil
4,Output sumber dan final identik,True,Berhasil
5,Artefak sumber tidak berubah,True,Berhasil
6,Seluruh file release tersedia,True,Berhasil
7,Status model tidak blocked,True,Berhasil
8,Training dijalankan,False,Sesuai
9,Bobot diperbarui,False,Sesuai


Seluruh pemeriksaan finalisasi berhasil.


In [27]:
SOURCE_HASH_TABLE_PATH = (
    FINALIZATION_OUTPUT_DIR
    / "source_artifact_hashes.csv"
)

RELEASE_HASH_TABLE_PATH = (
    FINALIZATION_OUTPUT_DIR
    / "release_artifact_hashes.csv"
)

SOURCE_INTEGRITY_PATH = (
    FINALIZATION_OUTPUT_DIR
    / "source_integrity_checks.csv"
)

FINAL_CHECK_PATH = (
    FINALIZATION_OUTPUT_DIR
    / "finalization_checks.csv"
)


source_hash_df.to_csv(
    SOURCE_HASH_TABLE_PATH,
    index=False,
    encoding="utf-8",
)

release_hash_df.to_csv(
    RELEASE_HASH_TABLE_PATH,
    index=False,
    encoding="utf-8",
)

source_integrity_df.to_csv(
    SOURCE_INTEGRITY_PATH,
    index=False,
    encoding="utf-8",
)

final_check_df.to_csv(
    FINAL_CHECK_PATH,
    index=False,
    encoding="utf-8",
)


print("Tabel audit finalisasi berhasil disimpan:")
print(f"- {SOURCE_HASH_TABLE_PATH.name}")
print(f"- {RELEASE_HASH_TABLE_PATH.name}")
print(f"- {SOURCE_INTEGRITY_PATH.name}")
print(f"- {FINAL_CHECK_PATH.name}")

Tabel audit finalisasi berhasil disimpan:
- source_artifact_hashes.csv
- release_artifact_hashes.csv
- source_integrity_checks.csv
- finalization_checks.csv


In [28]:
FINALIZATION_SUMMARY_PATH = (
    FINALIZATION_OUTPUT_DIR
    / "model_finalization_summary.json"
)


finalization_summary = {
    "created_at": (
        FINALIZED_AT
    ),

    "stage": (
        "14_finalisasi_model"
    ),

    "finalization_run_id": (
        FINALIZATION_RUN_ID
    ),

    "release": {
        "name": RELEASE_NAME,
        "version": RELEASE_VERSION,
        "status": RELEASE_STATUS,
        "directory": str(
            RELEASE_DIR.relative_to(
                PROJECT_DIR
            )
        ),
    },

    "model": {
        "model_name": MODEL_NAME,
        "num_classes": NUM_CLASSES,
        "class_to_idx": CLASS_TO_IDX,
        "input_shape": list(
            INPUT_SHAPE
        ),
    },

    "source_checkpoint": {
        "path": str(
            SOURCE_CHECKPOINT_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "sha256": (
            SOURCE_CHECKPOINT_SHA256
        ),
    },

    "final_checkpoint": {
        "path": str(
            FINAL_MODEL_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "sha256": calculate_sha256(
            FINAL_MODEL_PATH
        ),
        "size_bytes": int(
            FINAL_MODEL_PATH.stat().st_size
        ),
    },

    "selected_runs": {
        "evaluation": str(
            EVALUATION_RUN_DIR.relative_to(
                PROJECT_DIR
            )
        ),
        "error_analysis": str(
            ERROR_ANALYSIS_RUN_DIR.relative_to(
                PROJECT_DIR
            )
        ),
    },

    "acceptance": (
        acceptance_evaluation
    ),

    "test_metrics": (
        test_metrics
    ),

    "verification": {
        "source_checkpoint_strict_load": True,
        "final_checkpoint_strict_load": True,
        "source_and_final_output_equal": (
            SOURCE_AND_FINAL_OUTPUT_EQUAL
        ),
        "source_artifacts_unchanged": bool(
            source_integrity_df[
                "tidak_berubah"
            ].all()
        ),
        "all_final_checks_passed": True,
    },

    "actions": {
        "training_performed": False,
        "model_weights_updated": False,
        "dataset_loaded": False,
        "test_inference_performed": False,
        "synthetic_smoke_test_performed": True,
    },

    "output_artifacts": {
        "final_checkpoint": str(
            FINAL_MODEL_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "model_config": str(
            FINAL_MODEL_CONFIG_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "eval_transform": str(
            FINAL_EVAL_TRANSFORM_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "class_mapping": str(
            FINAL_CLASS_MAPPING_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "model_card": str(
            MODEL_CARD_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "model_manifest": str(
            MODEL_MANIFEST_PATH.relative_to(
                PROJECT_DIR
            )
        ),
        "torchscript": (
            str(
                TORCHSCRIPT_PATH.relative_to(
                    PROJECT_DIR
                )
            )
            if TORCHSCRIPT_EXPORTED
            else None
        ),
    },
}


write_json(
    finalization_summary,
    FINALIZATION_SUMMARY_PATH,
)


print("Ringkasan finalisasi berhasil disimpan:")
print(
    FINALIZATION_SUMMARY_PATH.relative_to(
        PROJECT_DIR
    )
)

Ringkasan finalisasi berhasil disimpan:
logs\resnet50_finalization_20260827_001248_921070\model_finalization_summary.json


In [29]:
FINALIZATION_REPORT_PATH = (
    FINALIZATION_OUTPUT_DIR
    / "model_finalization_report.md"
)


final_checkpoint_sha256 = (
    calculate_sha256(
        FINAL_MODEL_PATH
    )
)


finalization_report = f"""# Laporan Finalisasi Model

## Identitas Release

- Nama release: `{RELEASE_NAME}`
- Versi: `{RELEASE_VERSION}`
- Status: `{RELEASE_STATUS}`
- Tanggal finalisasi: `{FINALIZED_AT}`

## Model

- Arsitektur: `{MODEL_NAME}`
- Jumlah kelas: {NUM_CLASSES}
- Input shape: `{INPUT_SHAPE}`
- Dropout rate: `{DROPOUT_RATE}`

## Checkpoint

- Checkpoint sumber: `{SOURCE_CHECKPOINT_PATH.relative_to(PROJECT_DIR)}`
- SHA-256 checkpoint sumber: `{SOURCE_CHECKPOINT_SHA256}`
- Checkpoint final: `{FINAL_MODEL_PATH.relative_to(PROJECT_DIR)}`
- SHA-256 checkpoint final: `{final_checkpoint_sha256}`

## Sumber Evaluasi

- Run evaluasi: `{EVALUATION_RUN_DIR.relative_to(PROJECT_DIR)}`
- Run analisis kesalahan: `{ERROR_ANALYSIS_RUN_DIR.relative_to(PROJECT_DIR)}`
- Status kriteria penerimaan: {acceptance_evaluation.get("status", "Tidak tersedia")}

## Verifikasi

- Checkpoint sumber dimuat dengan strict mode: Berhasil
- Checkpoint final dimuat dengan strict mode: Berhasil
- Output sumber dan final identik: {SOURCE_AND_FINAL_OUTPUT_EQUAL}
- Artefak sumber tidak berubah: {bool(source_integrity_df["tidak_berubah"].all())}
- TorchScript diekspor: {TORCHSCRIPT_EXPORTED}
- TorchScript diverifikasi: {TORCHSCRIPT_VERIFIED}

## Kontrol Metodologis

- Training dijalankan: Tidak
- Bobot model diperbarui: Tidak
- Dataset train dibaca: Tidak
- Dataset validation dibaca: Tidak
- Dataset test dibaca ulang: Tidak
- Test digunakan untuk memilih model: Tidak
- Smoke test tensor sintetis: Ya

## Hasil

Paket model final tersimpan pada:

`{RELEASE_DIR.relative_to(PROJECT_DIR)}`

Model memiliki status `{RELEASE_STATUS}`. Jika statusnya `research_candidate`,
model dapat digunakan untuk eksperimen dan pengembangan sistem, tetapi belum
boleh dinyatakan memenuhi standar production sebelum kriteria penerimaan
ditetapkan dan dinilai secara independen.
"""


FINALIZATION_REPORT_PATH.write_text(
    finalization_report,
    encoding="utf-8",
)


print(finalization_report)

# Laporan Finalisasi Model

## Identitas Release

- Nama release: `resnet50_gcd_v1.0.0`
- Versi: `1.0.0`
- Status: `research_candidate`
- Tanggal finalisasi: `2026-08-27T00:12:48+07:00`

## Model

- Arsitektur: `resnet50`
- Jumlah kelas: 7
- Input shape: `(3, 224, 224)`
- Dropout rate: `0.3`

## Checkpoint

- Checkpoint sumber: `models\resnet50_gcd_best.pth`
- SHA-256 checkpoint sumber: `70f2fbdd1aa783c10d935ccdc8a338f2521986156a14793d472d7a0a0e79b5c0`
- Checkpoint final: `models\releases\resnet50_gcd_v1.0.0\resnet50_gcd_final.pth`
- SHA-256 checkpoint final: `d319252c0237478770760ed00b89f89de3196133db1270c030070c6d00234d54`

## Sumber Evaluasi

- Run evaluasi: `logs\resnet50_evaluation_20260826_233246_842963`
- Run analisis kesalahan: `logs\resnet50_error_analysis_20260826_234013_581950`
- Status kriteria penerimaan: Belum dinilai karena kriteria penerimaan belum ditetapkan.

## Verifikasi

- Checkpoint sumber dimuat dengan strict mode: Berhasil
- Checkpoint final dimuat dengan strict

In [30]:
final_artifact_paths = [
    FINAL_MODEL_PATH,
    FINAL_MODEL_CONFIG_PATH,
    FINAL_EVAL_TRANSFORM_PATH,
    FINAL_CLASS_MAPPING_PATH,
    MODEL_CARD_PATH,
    MODEL_MANIFEST_PATH,
    SOURCE_HASH_TABLE_PATH,
    RELEASE_HASH_TABLE_PATH,
    SOURCE_INTEGRITY_PATH,
    FINAL_CHECK_PATH,
    FINALIZATION_SUMMARY_PATH,
    FINALIZATION_REPORT_PATH,
]


if TORCHSCRIPT_EXPORTED:
    final_artifact_paths.append(
        TORCHSCRIPT_PATH
    )


final_artifact_df = pd.DataFrame([
    {
        "Nama file": path.name,
        "Lokasi": str(
            path.relative_to(
                PROJECT_DIR
            )
        ),
        "Ukuran MB": (
            path.stat().st_size
            / (1024 ** 2)
        ),
        "Ada": path.is_file(),
    }
    for path in final_artifact_paths
])


display(
    final_artifact_df.style.format({
        "Ukuran MB": "{:.4f}",
    })
)

,Nama file,Lokasi,Ukuran MB,Ada
0,resnet50_gcd_final.pth,models\releases\resnet50_gcd_v1.0.0\resnet50_gcd_final.pth,90.0313,True
1,model_config.json,models\releases\resnet50_gcd_v1.0.0\model_config.json,0.0021,True
2,eval_transform.json,models\releases\resnet50_gcd_v1.0.0\eval_transform.json,0.0007,True
3,class_mapping.json,models\releases\resnet50_gcd_v1.0.0\class_mapping.json,0.0004,True
4,MODEL_CARD.md,models\releases\resnet50_gcd_v1.0.0\MODEL_CARD.md,0.0026,True
5,model_manifest.json,models\releases\resnet50_gcd_v1.0.0\model_manifest.json,0.0045,True
6,source_artifact_hashes.csv,logs\resnet50_finalization_20260827_001248_921070\source_artifact_hashes.csv,0.0011,True
7,release_artifact_hashes.csv,logs\resnet50_finalization_20260827_001248_921070\release_artifact_hashes.csv,0.0007,True
8,source_integrity_checks.csv,logs\resnet50_finalization_20260827_001248_921070\source_integrity_checks.csv,0.0012,True
9,finalization_checks.csv,logs\resnet50_finalization_20260827_001248_921070\finalization_checks.csv,0.0005,True


In [31]:
print("=" * 78)
print("FINALISASI MODEL SELESAI")
print("=" * 78)

print(
    f"Model                    : "
    f"{MODEL_NAME}"
)

print(
    f"Release version          : "
    f"{RELEASE_VERSION}"
)

print(
    f"Release status           : "
    f"{RELEASE_STATUS}"
)

print(
    f"Jumlah kelas             : "
    f"{NUM_CLASSES}"
)

print(
    f"Input shape              : "
    f"{INPUT_SHAPE}"
)

print(
    f"Checkpoint sumber        : "
    f"{SOURCE_CHECKPOINT_PATH.relative_to(PROJECT_DIR)}"
)

print(
    f"Checkpoint final         : "
    f"{FINAL_MODEL_PATH.relative_to(PROJECT_DIR)}"
)

print(
    f"SHA-256 final            : "
    f"{final_checkpoint_sha256}"
)

print(
    f"Run evaluasi             : "
    f"{EVALUATION_RUN_DIR.relative_to(PROJECT_DIR)}"
)

print(
    f"Run analisis kesalahan   : "
    f"{ERROR_ANALYSIS_RUN_DIR.relative_to(PROJECT_DIR)}"
)

print(
    f"Output sumber-final sama : "
    f"{SOURCE_AND_FINAL_OUTPUT_EQUAL}"
)

print(
    f"TorchScript              : "
    f"{TORCHSCRIPT_EXPORTED}"
)

print(
    f"Folder release           : "
    f"{RELEASE_DIR.relative_to(PROJECT_DIR)}"
)

print(
    f"Folder laporan           : "
    f"{FINALIZATION_OUTPUT_DIR.relative_to(PROJECT_DIR)}"
)

print("Training dijalankan      : Tidak")
print("Bobot diperbarui         : Tidak")
print("Dataset dibaca           : Tidak")
print("Test inference diulang   : Tidak")

print("=" * 78)

FINALISASI MODEL SELESAI
Model                    : resnet50
Release version          : 1.0.0
Release status           : research_candidate
Jumlah kelas             : 7
Input shape              : (3, 224, 224)
Checkpoint sumber        : models\resnet50_gcd_best.pth
Checkpoint final         : models\releases\resnet50_gcd_v1.0.0\resnet50_gcd_final.pth
SHA-256 final            : d319252c0237478770760ed00b89f89de3196133db1270c030070c6d00234d54
Run evaluasi             : logs\resnet50_evaluation_20260826_233246_842963
Run analisis kesalahan   : logs\resnet50_error_analysis_20260826_234013_581950
Output sumber-final sama : True
TorchScript              : False
Folder release           : models\releases\resnet50_gcd_v1.0.0
Folder laporan           : logs\resnet50_finalization_20260827_001248_921070
Training dijalankan      : Tidak
Bobot diperbarui         : Tidak
Dataset dibaca           : Tidak
Test inference diulang   : Tidak


## Interpretasi Hasil Finalisasi

### Checkpoint final

Checkpoint final berisi bobot model dan metadata yang diperlukan untuk
inferensi. State optimizer dan scheduler tidak disertakan karena hanya
dibutuhkan untuk melanjutkan training.

### Hash SHA-256

Hash digunakan untuk memastikan file model tidak berubah setelah proses
evaluasi dan finalisasi. Perubahan satu byte pada file akan menghasilkan nilai
hash yang berbeda.

### Smoke test

Smoke test memastikan arsitektur dapat menerima tensor dengan dimensi yang
ditentukan dan menghasilkan logits sebanyak jumlah kelas. Smoke test tidak
mengukur accuracy karena tidak menggunakan data berlabel.

### Status approved

Status `approved` diberikan jika seluruh kriteria penerimaan yang sebelumnya
telah ditetapkan pada notebook evaluasi terpenuhi.

### Status research_candidate

Status `research_candidate` berarti belum ada nilai ambang formal untuk
menentukan kelayakan model. Model masih dapat digunakan sebagai hasil penelitian,
prototype, atau eksperimen lanjutan, tetapi tidak boleh dinyatakan sebagai model
production yang telah lolos penerimaan.

### Status blocked

Status `blocked` berarti sedikitnya satu kriteria penerimaan tidak terpenuhi.
Notebook menghentikan proses sebelum model dikemas.

### Hubungan dengan prediksi citra baru

Hasil `13_prediksi_citra_baru.ipynb` tidak digunakan untuk memilih atau mengubah
model final. Citra baru mungkin tidak memiliki ground truth dan tidak dapat
digunakan sebagai dasar perhitungan performa model.

# Kesimpulan

Notebook finalisasi telah membentuk paket model klasifikasi jenis awan yang
terdiri atas checkpoint final, konfigurasi model, pipeline preprocessing,
pemetaan kelas, model manifest, dan model card.

Checkpoint final berasal dari checkpoint terbaik yang sebelumnya dipilih
menggunakan validation loss. Konsistensi checkpoint diperiksa terhadap metadata
evaluasi dan analisis kesalahan. Model final juga dimuat kembali menggunakan
`strict=True` dan diverifikasi melalui smoke test menggunakan tensor sintetis.

Selama finalisasi:

- training tidak dijalankan;
- bobot model tidak diperbarui;
- dataset train, validation, dan test tidak dibaca;
- inference pada test split tidak diulang;
- test set tidak digunakan untuk memilih model;
- artefak sumber tidak dimodifikasi.

Paket model final tersimpan di dalam `models/releases/`, sedangkan laporan,
hash, dan tabel audit finalisasi tersimpan di dalam folder run pada `logs/`.

Tahap berikutnya adalah mengimplementasikan proses inferensi yang stabil pada
`src/predict.py` menggunakan checkpoint final, `model_config.json`,
`class_mapping.json`, dan `eval_transform.json` dari paket release.